# Preference alignment end to end: data → reward model → PPO, GRPO, DPO

Chapters 10–13 · Day 15 · **DGX Spark GPU first**

This notebook gathers the preference-learning ideas from our SLP3 §8.3–8.4 discussions into one
flow, with real alignment data and the real **Qwen3-0.6B** model. Each part builds on the last.

| Part | Question | Discussions in the [learning note](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
|---|---|---|
| A · Data | What does preference data look like, for chat and for agents? How noisy is it? | D1, D3, D19 |
| B · Bradley–Terry | How does a score gap become a probability and a loss? | D4–D7, D9–D10 |
| C · Reward model | What is a reward model, physically? How is it trained, and what does it learn? | D8, D11, D15–D16 |
| D · Phase 2 data | What does RL train on, if not labeled pairs? | D12, D19–D20 |
| E · PPO | What are the policy, reference, reward model and value model? How do rewards, values, GAE and clipping fit together? | D21–D25 |
| F · GRPO | What replaces the value model? | D21, D23 |
| G · DPO | How do you learn from the same pairs with no reward model? | D14, D19, D21 |

**Data.** Anthropic's [HH-RLHF](https://huggingface.co/datasets/Anthropic/hh-rlhf) (MIT): human
comparisons of two assistant replies, from Bai et al. 2022. We use only its `helpful-base` subset.
The `harmless-base` subset is red-teaming data with offensive text, so it isn't printed here.
[UltraFeedback binarized](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized)
(MIT): AI feedback, where GPT-4 rated 4 answers per prompt and the best and a worse one became a
pair; the notebook uses its 2,000-row `test_prefs` file for statistics only. The agent trajectories
in Part A are authored for this notebook.

**Environment.** Use the `Python (DGX Spark Native)` kernel with CUDA. Qwen3-0.6B must be in the
local Hugging Face cache. The two dataset files (about 25 MB) download once into the same cache on
the first run. Total run time is about 20–25 minutes on the Spark, most of it reward-model
training (Part C).

**Scale and limits.** The reward model trains on 3,000 pairs, PPO and GRPO run for a few dozen
iterations, and DPO for 200 steps. They show *mechanics*: what each model reads and outputs, what
each loss does, how many models are in memory. They don't produce an aligned assistant, and a
change in reward-model score isn't a change in real quality. GPU results vary slightly between
runs; checks use tolerances.

In [ ]:
import copy, gc, gzip, json, math, random, re, time, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F
import transformers
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this notebook is GPU first."
DEVICE = "cuda"
MODEL = "Qwen/Qwen3-0.6B"
transformers.logging.set_verbosity_error()
random.seed(0); torch.manual_seed(0)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.family": "DejaVu Sans",
                     "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GRAY, GREEN, RED = "#2563EB", "#EA580C", "#94A3B8", "#16A34A", "#DC2626"

tok = AutoTokenizer.from_pretrained(MODEL, local_files_only=True)
IM_END = tok.convert_tokens_to_ids("<|im_end|>")
PAD = tok.pad_token_id

def load_chat_model():
    return AutoModelForCausalLM.from_pretrained(MODEL, local_files_only=True, dtype=torch.bfloat16).to(DEVICE)

def free():
    """Release GPU memory after `del`-ing models at the top level."""
    gc.collect(); torch.cuda.empty_cache()

def gib(n_bytes):
    return n_bytes / 2**30

print(f"torch {torch.__version__} | transformers {transformers.__version__} | {torch.cuda.get_device_name()}")

---

# Part A · What preference data looks like

A preference judgment is a **comparison**: for the same prompt, which of two responses is better
under some criterion (D1). A dataset of them is a list of `(prompt, chosen, rejected)` triples. Real
datasets store this in different shapes, and a large part of the engineering is turning them into
clean triples.

In [ ]:
def hh_rows(split):
    path = hf_hub_download("Anthropic/hh-rlhf", f"helpful-base/{split}.jsonl.gz", repo_type="dataset")
    with gzip.open(path, "rt") as f:
        return [json.loads(line) for line in f]

hh_train_raw, hh_test_raw = hh_rows("train"), hh_rows("test")
raw = next(r for r in hh_train_raw if r["chosen"].startswith("\n\nHuman: I want to plant my own garlic"))
print(f"{len(hh_train_raw)} train and {len(hh_test_raw)} test records; keys: {list(raw)}\n")
print("CHOSEN:", raw["chosen"], "\n")
print("REJECTED:", raw["rejected"])

### Prediction

## Predict 1 · Where is the prompt?

An HH-RLHF record has only two fields, `chosen` and `rejected`, each a full transcript. There is no `prompt` field. Where is the prompt, and what must be true of a record for it to become a clean `(prompt, chosen, rejected)` triple? Guess what fraction of records fail that test.

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
def parse_transcript(text):
    """'\\n\\nHuman: ...\\n\\nAssistant: ...' -> chat messages."""
    parts = re.split(r"\n\n(Human|Assistant): ", text)[1:]
    return [{"role": "user" if who == "Human" else "assistant", "content": said.strip()}
            for who, said in zip(parts[0::2], parts[1::2])]

def to_pair(row):
    """Return {prompt, chosen, rejected}, or None if the two transcripts don't differ only in the last reply."""
    c, r = parse_transcript(row["chosen"]), parse_transcript(row["rejected"])
    if len(c) < 2 or c[:-1] != r[:-1] or c[-1]["role"] != "assistant" or r[-1]["role"] != "assistant":
        return None
    if c[-1]["content"] == r[-1]["content"]:
        return None
    return {"prompt": c[:-1], "chosen": c[-1]["content"], "rejected": r[-1]["content"]}

pair = to_pair(raw)
print(json.dumps(pair, indent=1, ensure_ascii=False))

train_pairs = [p for p in map(to_pair, hh_train_raw) if p]
test_pairs = [p for p in map(to_pair, hh_test_raw) if p]
dropped = 1 - len(train_pairs) / len(hh_train_raw)
print(f"\nclean pairs: {len(train_pairs)} train, {len(test_pairs)} test | dropped {dropped:.1%} of train records")
assert pair["prompt"] == [{"role": "user", "content": "I want to plant my own garlic. Can you tell me how to do that?"}]
assert 0 < dropped < 0.2

### Reference explanation

### Reference answer and mechanism

The prompt is the **shared prefix** of the two transcripts. Only the final assistant reply differs:
in this record, a concrete planting guide (chosen) vs. "Here are some photos" with no photos
(rejected). A record becomes a clean triple only if the two transcripts agree on every turn but the
last, both end with an assistant reply, and the replies differ. The rest are dropped, a few percent
here: transcripts that differ earlier, end on a human turn, or are identical.

Prompts can be **multi-turn**: `prompt` is a list of messages, and the judged reply is the next
assistant turn. The labeler saw both replies and clicked the better one. That click is the whole
label (D10): "chosen beat rejected", no score and no reason.

UltraFeedback stores the same idea differently. A GPT-4 judge rated four answers per prompt from
1 to 10. The binarized version keeps the highest-rated answer as `chosen` and a random lower one as
`rejected`, with both ratings. This is **AI feedback** (RLAIF, D18), not human clicks.

In [ ]:
uf_path = hf_hub_download("HuggingFaceH4/ultrafeedback_binarized", "data/test_prefs-00000-of-00001.parquet",
                          repo_type="dataset")
uf = pd.read_parquet(uf_path)
print(f"{len(uf)} rows; columns: {list(uf.columns)}\n")
row = uf.iloc[3]
print("PROMPT:", textwrap.shorten(row.prompt, 200))
print(f"CHOSEN   (judge score {row.score_chosen}):", textwrap.shorten(row.chosen[-1]["content"], 200))
print(f"REJECTED (judge score {row.score_rejected}):", textwrap.shorten(row.rejected[-1]["content"], 200))

### Prediction

## Predict 2 · How clean is the label?

In UltraFeedback, the `chosen` answer is the one the judge scored highest. In what fraction of pairs do you expect the two scores to be **equal**, so that the pair says nothing about which answer is better? How large is a typical score margin?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
margin = (uf.score_chosen - uf.score_rejected).to_numpy()
ties = float((margin == 0).mean())
print(f"ties: {ties:.1%} | margin < 1: {(margin < 1).mean():.1%} | median margin: {np.median(margin):.2f}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(margin, bins=np.arange(-0.25, 9.5, 0.5), color=BLUE, edgecolor="white")
ax.set_xlabel("judge score of chosen − judge score of rejected"); ax.set_ylabel("pairs")
ax.set_title("UltraFeedback test_prefs: how much better is 'chosen'?")
plt.tight_layout(); plt.show()
assert 0 <= ties < 0.2

### Reference explanation

### Reference answer and mechanism

About **one pair in seven is an exact tie** (13.6% in this file): the binarization still labeled one
answer "chosen", so those labels are coin flips. About a third of pairs have a margin below 1 point
on a 10-point scale, where a different judge, or the same judge asked again, might well disagree.
This is the label noise of D3, measured.

Two consequences. First, a reward model can't reach 100% agreement with these labels, and even
large reward models agree with held-out human labels only about 65–75% of the time. Second,
Bradley–Terry already expects this: a small gap means a probability near 0.5, not a certainty
(Part B).

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Distribution of judge-score margins in UltraFeedback pairs](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-01.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Prediction

## Predict 3 · Is 'chosen' longer?

Across both datasets, is the chosen answer usually **longer** than the rejected one? If it is, what might a reward model trained on these pairs learn, besides quality?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
def words(text):
    return len(text.split())

hh_gap = np.array([words(p["chosen"]) - words(p["rejected"]) for p in train_pairs])
uf_gap = np.array([words(c[-1]["content"]) - words(r[-1]["content"]) for c, r in zip(uf.chosen, uf.rejected)])
for name, gap in [("HH-RLHF helpful (human)", hh_gap), ("UltraFeedback (GPT-4 judge)", uf_gap)]:
    print(f"{name:28s} chosen longer: {(gap > 0).mean():.1%} | shorter: {(gap < 0).mean():.1%} | "
          f"median gap: {np.median(gap):+.0f} words")

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=False)
for ax, (name, gap) in zip(axes, [("HH-RLHF helpful (human)", hh_gap), ("UltraFeedback (GPT-4 judge)", uf_gap)]):
    ax.hist(np.clip(gap, -300, 300), bins=60, color=ORANGE, edgecolor="white")
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(name); ax.set_xlabel("words(chosen) − words(rejected), clipped at ±300")
axes[0].set_ylabel("pairs")
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

In both datasets the chosen answer is longer more often than it is shorter. Longer answers often
*are* more helpful, but a reward model can't tell "more complete" from "more words". If length and
preference go together in the data, length becomes a cheap feature for the reward model to pick up
(D15: it learns the judges' taste, habits included). Part C measures whether ours did. This is the
starting point of length-based reward hacking (plan P2; Chapter 14 §14.5).

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Word-length gap between chosen and rejected answers in two datasets](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-02.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Agent trajectories

For an agent, each side of a pair is a whole **trajectory**: the model's thoughts and tool calls,
the tool results, and the final answer (D19). Below are two authored trajectories for one task,
rendered with Qwen3's own chat template and tool format. The chosen one calls the declared tool
with valid arguments and uses its result; the rejected one calls a tool that doesn't exist and then
invents an answer.

In [ ]:
tools = [{"type": "function", "function": {
    "name": "get_weather", "description": "Daily forecast for a city",
    "parameters": {"type": "object", "properties": {"city": {"type": "string"}, "date": {"type": "string"}},
                   "required": ["city", "date"]}}}]
task = [{"role": "user", "content": "What's the weather in Paris tomorrow (2026-10-28), in °F?"}]

def call(name, **args):
    return {"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": name, "arguments": args}}]}

chosen_traj = task + [call("get_weather", city="Paris", date="2026-10-28"),
                      {"role": "tool", "content": '{"high_c": 14, "low_c": 8}'},
                      {"role": "assistant", "content": "Tomorrow in Paris: high about 57°F, low about 46°F."}]
rejected_traj = task + [call("get_forecast", location="Paris"),
                        {"role": "tool", "content": '{"error": "unknown tool get_forecast"}'},
                        {"role": "assistant", "content": "It will be sunny and 75°F in Paris tomorrow."}]
agent_pair = {"tools": tools, "chosen": chosen_traj, "rejected": rejected_traj}

text = tok.apply_chat_template(chosen_traj, tools=tools, tokenize=False, enable_thinking=False)
print(text[text.index("<|im_start|>user"):])

### Prediction

## Predict 4 · Which tokens of a trajectory are the model's?

If a policy is trained on this chosen trajectory, which tokens is it graded on: the user turn, the tool call, the tool response, the final answer? Note that Qwen3 places tool responses inside a `user` turn.

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
HEADER = tok.encode("<|im_start|>assistant\n", add_special_tokens=False)

def model_written_mask(ids):
    """1 for tokens inside assistant turns (through <|im_end|>), 0 elsewhere: system, user and tool text."""
    mask, i = [0] * len(ids), 0
    while i < len(ids):
        if ids[i:i + len(HEADER)] == HEADER:
            j = i + len(HEADER)
            while j < len(ids) and ids[j] != IM_END:
                mask[j] = 1; j += 1
            if j < len(ids):
                mask[j] = 1
            i = j
        i += 1
    return mask

ids = tok.apply_chat_template(chosen_traj, tools=tools, enable_thinking=False, return_dict=True)["input_ids"]
mask = model_written_mask(list(ids))
segments, current, state = [], [], mask[0]
for t, m in zip(ids, mask):
    if m != state:
        segments.append((state, tok.decode(current))); current, state = [], m
    current.append(t)
segments.append((state, tok.decode(current)))
for m, s in segments[1:]:
    print(("GRADED   " if m else "context  ") + repr(s[-110:]))
graded = tok.decode([t for t, m in zip(ids, mask) if m])
assert "get_weather" in graded and "57°F" in graded and "high_c" not in graded

### Reference explanation

### Reference answer and mechanism

The policy is graded only on what **it** wrote: the `<tool_call>` block and the final answer, each
through `<|im_end|>`. The task, the tool definitions and the `<tool_response>` are context. The tool
response was produced by the environment, so the model shouldn't be pushed to "predict" it. Training
on it would teach the model to imagine tool results, the exact failure of the rejected trajectory.
This is the `-100` label mask from instruction tuning, applied to trajectories.

The empty `<think>` block on the last turn is template scaffolding (as in Day 12 · 03); a real
pipeline must decide consistently whether it's graded. For a preference pair, the same mask is
applied to both trajectories, and the label can come from a human, an LLM judge, or the **outcome**:
did the forecast match? That last case is a verifier, i.e. RLVR.

---

# Part B · Bradley–Terry: from a score gap to a loss

The reward model will output one number per (prompt, response). Bradley–Terry turns two numbers into
a probability: `P(chosen ≻ rejected) = σ(r_chosen − r_rejected)` (D4). Training maximizes the
probability of the observed label, so the loss is `−log σ(r_chosen − r_rejected)` (D9).

### Prediction

## Predict 5 · Gap, probability, loss, gradient

For score gaps of −2, 0, 1 and 4, what are the probability and the loss? What happens if you add 10 to both scores? And when the model already prefers the chosen answer by a wide margin, how big is the gradient?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
r_chosen = torch.tensor([0.0, 1.0, 2.0, 5.0], requires_grad=True)
r_rejected = torch.tensor([2.0, 1.0, 1.0, 1.0])
gap = r_chosen - r_rejected
prob = torch.sigmoid(gap)
loss = -F.logsigmoid(gap)
bce = F.binary_cross_entropy_with_logits(gap, torch.ones_like(gap), reduction="none")
loss.sum().backward()
for g, p, l, d in zip(gap.tolist(), prob.tolist(), loss.tolist(), r_chosen.grad.tolist()):
    print(f"gap {g:+.0f}: P(chosen wins) {p:.3f} | loss {l:.3f} | d loss / d r_chosen {d:+.3f}")
torch.testing.assert_close(loss, bce)
shifted = -F.logsigmoid((r_chosen + 10) - (r_rejected + 10))
torch.testing.assert_close(shifted, loss)
torch.testing.assert_close(r_chosen.grad, -(1 - prob.detach()))

### Reference explanation

### Reference answer and mechanism

- **Gap → probability.** A gap of 0 gives 0.5; +1 gives 0.73; +4 gives 0.98; −2 gives 0.12.
- **Loss.** `−log σ(gap)` is exactly binary cross-entropy with target 1 (the assertion), where the
  "logit" is the score gap. The ground truth is the judge's choice; the prediction is the model's
  probability for it (D10).
- **Only gaps matter.** Adding 10 to both scores changes nothing, so a reward model's absolute score
  has no meaning; only differences on the same prompt do (D15).
- **Gradient = −(1 − σ(gap)).** Wrong or unsure pairs get pushed hardest. A pair the model already
  gets right by a wide margin (gap +4) gets almost no gradient, so training focuses on the pairs it
  doesn't yet understand.

In [ ]:
g = torch.linspace(-6, 6, 241)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].plot(g, torch.sigmoid(g), color=BLUE); axes[0].set_title("P(chosen ≻ rejected) = σ(gap)")
axes[1].plot(g, -F.logsigmoid(g), color=ORANGE); axes[1].set_title("loss = −log σ(gap)")
axes[2].plot(g, -(1 - torch.sigmoid(g)), color=RED); axes[2].set_title("d loss / d r_chosen = −(1 − σ(gap))")
for ax in axes:
    ax.axvline(0, color=GRAY, linewidth=0.8); ax.set_xlabel("score gap r_chosen − r_rejected")
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Bradley-Terry probability, pairwise loss and its gradient against the score gap](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-03.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

---

# Part C · The reward model: an LLM with a one-number head

A reward model is a **separate copy** of a pretrained LLM whose next-token output layer is replaced by
a linear layer with **one output** (D15–D16). It reads a prompt and a full response and outputs a
score. It never generates text (D11).

In [ ]:
rm = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1, local_files_only=True,
                                                        dtype=torch.bfloat16).to(DEVICE)
rm.config.pad_token_id = PAD
n_params = sum(p.numel() for p in rm.parameters())
print(rm.score)
print(f"parameters: {n_params / 1e6:.1f} M | new head: {rm.score.weight.numel()} weights (hidden size × 1)")
assert rm.score.out_features == 1

### Reference explanation

### Reference answer and mechanism

`AutoModelForSequenceClassification` with `num_labels=1` loads Qwen3-0.6B's transformer, **drops**
the language-model head (1024 → 151,936 vocabulary logits) and adds `score`, a 1024 → 1 linear
layer, randomly initialized. Everything the transformer learned in pretraining is kept, so the
reward model starts out understanding language and only has to learn *taste* (D15). The head adds
just 1,024 weights.

### Prediction

## Predict 6 · Where does the single number come from?

The transformer produces a hidden vector at **every** token position. The reward model returns one number per sequence. Which position's vector does it use, and does the head produce a number at the other positions too?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
def render(prompt, answer):
    """A full conversation as text, with Qwen3's template and no thinking."""
    text = tok.apply_chat_template(prompt + [{"role": "assistant", "content": answer}], tokenize=False,
                                   enable_thinking=False)
    return text.removesuffix("\n")          # end on <|im_end|>, exactly where a sampled answer ends

b = tok([render(pair["prompt"], pair["chosen"])], return_tensors="pt").to(DEVICE)
with torch.no_grad():
    hidden = rm.model(**b).last_hidden_state                  # [1, T, 1024]
    per_position = rm.score(hidden).squeeze(-1).float()       # [1, T]: one number at every position
    pooled = rm(**b).logits.float()                           # [1, 1]: what the reward model returns
print(f"tokens: {b.input_ids.shape[1]} | per-position outputs: {tuple(per_position.shape)} | returned: {tuple(pooled.shape)}")
print(f"last position: {per_position[0, -1]:.4f} | returned score: {pooled[0, 0]:.4f} | last token: {tok.decode(b.input_ids[0, -1:])!r}")
torch.testing.assert_close(per_position[0, -1], pooled[0, 0], rtol=1e-3, atol=1e-3)
assert b.input_ids[0, -1].item() == IM_END

### Reference explanation

### Reference answer and mechanism

The head runs at every position, but the reward model **returns only the last token's** number,
here `<|im_end|>`: the position that has read the whole prompt and the whole response. (`render`
drops the template's trailing newline so that training texts end exactly where a sampled answer
ends in Part E; otherwise the score would be read at different tokens in training and in RL.) (With padding, it picks the
last non-pad token.) Through attention, that one position can draw on the entire sequence.

Keep the per-position outputs in mind. A **value model** (Part E) uses exactly this: the same kind
of network, but read at every position, giving a prediction for each partial response.

### Prediction

## Predict 7 · How good is an untrained reward model?

Before any training, on held-out HH pairs, how often will the reward model give the chosen answer the higher score?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
RM_MAXLEN = 320

def rm_scores(model, prompts, answers, bs=16):
    out = []
    for i in range(0, len(answers), bs):
        texts = [render(p, a) for p, a in zip(prompts[i:i + bs], answers[i:i + bs])]
        enc = tok(texts, return_tensors="pt", padding=True, padding_side="right", truncation=True,
                  max_length=RM_MAXLEN).to(DEVICE)
        out.append(model(**enc).logits.squeeze(-1).float())
    return torch.cat(out)

@torch.no_grad()
def pair_accuracy(model, pairs):
    model.eval()
    s_c = rm_scores(model, [p["prompt"] for p in pairs], [p["chosen"] for p in pairs])
    s_r = rm_scores(model, [p["prompt"] for p in pairs], [p["rejected"] for p in pairs])
    return (s_c > s_r).float().mean().item(), s_c, s_r

rm_eval_pairs = test_pairs[:500]
acc_before, _, _ = pair_accuracy(rm, rm_eval_pairs)
print(f"held-out accuracy before training: {acc_before:.1%} on {len(rm_eval_pairs)} pairs")
assert 0.35 < acc_before < 0.65

### Reference explanation

### Reference answer and mechanism

About 50%: chance. The head is random, so its scores are unrelated to quality, even though the
transformer underneath understands the text. Understanding language and having a preference are
different things; only the second is learned here.

### Training: the pairwise loss on real pairs

Each step scores the chosen and rejected answers of 8 pairs (16 sequences in one forward pass) and
applies `−log σ(r_chosen − r_rejected)`. All weights train (full fine-tuning). This is the slow part
of the notebook, about 6–8 minutes.

In [ ]:
N_RM_TRAIN, RM_BS, RM_LR = 3000, 8, 1e-5
random.seed(0)
rm_train_pairs = random.sample(train_pairs[:20000], N_RM_TRAIN)   # train_pairs[20000:] stays unused by the reward model
opt = torch.optim.AdamW(rm.parameters(), lr=RM_LR, weight_decay=0.0)
rm.train(); rm_losses, t0 = [], time.time()
torch.cuda.reset_peak_memory_stats()
for i in range(0, N_RM_TRAIN, RM_BS):
    batch = rm_train_pairs[i:i + RM_BS]
    prompts = [p["prompt"] for p in batch] * 2
    scores = rm_scores(rm, prompts, [p["chosen"] for p in batch] + [p["rejected"] for p in batch], bs=2 * RM_BS)
    loss = -F.logsigmoid(scores[:len(batch)] - scores[len(batch):]).mean()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0)
    opt.step(); opt.zero_grad()
    rm_losses.append(loss.item())
    if len(rm_losses) % 75 == 0:
        print(f"step {len(rm_losses):4d} | mean loss (last 75) {np.mean(rm_losses[-75:]):.3f} | {time.time() - t0:.0f}s")
rm_peak = gib(torch.cuda.max_memory_allocated())
del opt; free()
rm.eval(); rm.requires_grad_(False)          # frozen from here on: it only grades
print(f"trained on {N_RM_TRAIN} pairs in {time.time() - t0:.0f}s | peak GPU memory {rm_peak:.1f} GiB")

### Prediction

## Predict 8 · Accuracy after training

After one pass over 3,000 pairs, what held-out accuracy do you expect? Keep Part A's label noise in mind: what is the best any model could do?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
acc_after, eval_sc, eval_sr = pair_accuracy(rm, rm_eval_pairs)
print(f"held-out accuracy: {acc_before:.1%} before → {acc_after:.1%} after")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
smooth = np.convolve(rm_losses, np.ones(25) / 25, mode="valid")
axes[0].plot(rm_losses, color=GRAY, alpha=0.4, linewidth=0.8, label="per step")
axes[0].plot(np.arange(len(smooth)) + 12, smooth, color=BLUE, label="25-step mean")
axes[0].axhline(math.log(2), color=RED, linestyle="--", linewidth=1, label="log 2 (chance)")
axes[0].set_xlabel("step"); axes[0].set_ylabel("pairwise loss"); axes[0].legend(); axes[0].set_title("reward-model training")
gaps = (eval_sc - eval_sr).cpu().numpy()
axes[1].hist(gaps, bins=40, color=BLUE, edgecolor="white"); axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("r(chosen) − r(rejected) on held-out pairs"); axes[1].set_title(f"held-out gaps: {acc_after:.0%} right of 0")
plt.tight_layout(); plt.show()
assert acc_after > acc_before + 0.05

### Reference explanation

### Reference answer and mechanism

Accuracy rises well above chance, to roughly the low-to-mid 60s in the reference run, and the loss
falls below `log 2 ≈ 0.693`, the loss of always answering 50/50. That's a typical result for HH: the
labels are noisy (Part A), and reward models far larger than ours reach only about 70% on this data.

The gap histogram shows what "accuracy" hides: many held-out pairs sit close to 0, where the model
is unsure, and a fair number are confidently wrong. Those mistakes are what a policy can later
exploit.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Reward-model training loss and held-out score gaps](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-04.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Prediction

## Predict 9 · What else did it learn?

Take 100 held-out chosen answers and append one friendly but empty sentence: *\"I hope this helps! Let me know if you have any other questions.\"* Will the reward model's score go up, go down or stay the same? Across held-out pairs, does a longer answer tend to get a higher score?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
FILLER = " I hope this helps! Let me know if you have any other questions."
probe = test_pairs[500:600]
with torch.no_grad():
    base = rm_scores(rm, [p["prompt"] for p in probe], [p["chosen"] for p in probe])
    padded = rm_scores(rm, [p["prompt"] for p in probe], [p["chosen"] + FILLER for p in probe])
delta = (padded - base).cpu().numpy()
print(f"filler sentence: score rises for {(delta > 0).mean():.0%} of answers | mean change {delta.mean():+.3f} "
      f"(held-out gap std {gaps.std():.3f})")

len_gap = np.array([words(p["chosen"]) - words(p["rejected"]) for p in rm_eval_pairs])
corr = np.corrcoef(len_gap, gaps)[0, 1]
print(f"correlation between word-length gap and score gap on held-out pairs: {corr:+.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].scatter(np.clip(len_gap, -200, 200), gaps, s=8, color=BLUE, alpha=0.5)
axes[0].axhline(0, color=GRAY, linewidth=0.8); axes[0].axvline(0, color=GRAY, linewidth=0.8)
axes[0].set_xlabel("words(chosen) − words(rejected), clipped"); axes[0].set_ylabel("score gap")
axes[0].set_title(f"length vs. score gap (r = {corr:+.2f})")
axes[1].hist(delta, bins=30, color=ORANGE, edgecolor="white"); axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("score(answer + filler) − score(answer)"); axes[1].set_title("effect of one empty sentence")
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

Two different answers, so read both numbers from your run.

- **Length.** In the reference run the score gap correlates with the length gap at about +0.4:
  the longer answer of a pair tends to get the higher score. That is partly legitimate (longer HH
  answers are often more complete) and partly a shortcut. This correlation alone can't separate the
  two; a controlled probe is needed, like the next one.
- **The filler sentence.** It *lowered* most scores: in the reference run about one answer in five
  went up, and the mean change was small and negative compared with the spread of held-out gaps. So this particular cheap trick
  doesn't fool this reward model. Whatever it learned about length, it isn't "more friendly
  words = better".

Both are the right kind of question to ask: **what features does the score respond to, besides
quality?** The reward model learned whatever predicted the labels in its 3,000 pairs (Part A), and
nothing guarantees that those features are only quality.

This is why a reward model is a **proxy** (D15). One probe that fails to fool it proves little: a
policy trained to maximize the score searches far more widely than any probe we write, and will
find whatever shortcut does exist, because a shortcut is easier than becoming more helpful. That's reward hacking
(plan P2); Chapter 14 measures it at scale.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Reward-model score against length, and the effect of an empty filler sentence](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-05.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

---

# Part D · Phase 2 data: prompts only

Reward-model training needed labeled pairs. Training the chat model with RL needs only **prompts**:
the chat model writes its own answers, and the frozen reward model labels them (D19–D20).

### Prediction

## Predict 10 · What is in the RL dataset?

List exactly what the RL training set contains. Where does the 'label' for each training example come from, and when is it produced?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
def first_question(row):
    turns = parse_transcript(row["chosen"])
    return turns[0]["content"] if turns and turns[0]["role"] == "user" else None

rl_prompts = [q for q in map(first_question, hh_train_raw[20000:]) if q and 15 < len(q) < 200]
eval_prompts = [q for q in map(first_question, hh_test_raw[600:]) if q and 15 < len(q) < 200][:64]
print(f"{len(rl_prompts)} RL training prompts, {len(eval_prompts)} evaluation prompts; type: {type(rl_prompts[0]).__name__}")
for q in rl_prompts[:3]:
    print(" ·", q)
assert all(isinstance(q, str) for q in rl_prompts)

### Reference explanation

### Reference answer and mechanism

The RL set is a list of **strings**: user questions, with no answers and no labels. They come from
the part of HH that the reward model never trained on, so the policy can't simply meet the reward
model's training examples. The label for each example is produced **during training**: the policy
writes an answer, and the frozen reward model scores it at that moment. Humans labeled pairs once,
in phase 1; their judgment now acts through the reward model on answers they never saw.

In [ ]:
policy = load_chat_model()
tok_prompt = lambda q: tok.apply_chat_template([{"role": "user", "content": q}], add_generation_prompt=True,
                                               tokenize=False, enable_thinking=False)
torch.manual_seed(1)
enc = tok([tok_prompt(eval_prompts[0])], return_tensors="pt").to(DEVICE)
with torch.no_grad():
    out = policy.generate(**enc, max_new_tokens=96, do_sample=True, temperature=1.0, top_k=0, top_p=1.0, pad_token_id=PAD)
answer = tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True)
score = rm_scores(rm, [[{"role": "user", "content": eval_prompts[0]}]], [answer])[0]
print("PROMPT:", eval_prompts[0]); print("POLICY:", textwrap.shorten(answer, 400)); print(f"REWARD MODEL: {score:.3f}")

That is one step of phase 2: **prompt → policy writes → reward model scores the whole answer → one
number**. Everything in Parts E and F is about turning that one number into an update of the
policy's token probabilities.

---

# Part E · PPO, component by component

PPO-based RLHF keeps four networks in memory (D24). `policy` is already loaded; the next cell adds
the other three.

### Prediction

## Predict 11 · Four networks: which are copies of which?

We'll create `ref` from the base chat model and `value` from the trained reward model. Before any PPO step, which pairs of networks have identical weights? Which of the four will be trained? And what is the shape of each one's output for a batch of B sequences of length T?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
ref = load_chat_model().eval().requires_grad_(False)       # frozen copy of the starting policy: the KL anchor
value = copy.deepcopy(rm).train().requires_grad_(True)     # separate network, initialized from the reward model

def weights_hash(model):
    return hash(tuple(round(float(p.detach().float().sum()), 3) for p in list(model.parameters())[:20]))

ids = tok([tok_prompt("Hello")], return_tensors="pt").to(DEVICE).input_ids
with torch.no_grad():
    shapes = {"policy": policy(ids).logits.shape, "ref": ref(ids).logits.shape,
              "reward model": rm(ids).logits.shape,
              "value": value.score(value.model(ids).last_hidden_state).squeeze(-1).shape}
rows = [("policy", "trained", "next-token logits [B, T, vocab]"), ("ref", "frozen", "next-token logits [B, T, vocab]"),
        ("reward model", "frozen", "one score per sequence [B, 1]"), ("value", "trained", "one number per position [B, T]")]
for name, status, what in rows:
    print(f"{name:13s} {status:8s} {what:34s} actual {tuple(shapes[name])}")
print("policy == ref weights:", weights_hash(policy) == weights_hash(ref),
      "| value == reward model weights:", weights_hash(value) == weights_hash(rm), "| value is rm:", value is rm)
assert weights_hash(policy) == weights_hash(ref) and weights_hash(value) == weights_hash(rm) and value is not rm

### Reference explanation

### Reference answer and mechanism

- `policy` and `ref` start **identical** (both are the chat model). Only `policy` trains; `ref`
  stays frozen so we can measure how far the policy has moved (the KL penalty).
- `value` starts as an identical **copy** of the reward model but is a separate object, and it
  trains. It is read at every position, giving `[B, T]`: a prediction of the final return from each
  partial response (D23–D24).
- The reward model is frozen and returns one number per sequence.

So the value model is **not** a snapshot of the policy (D25): it outputs numbers, not token
probabilities, and starts from the reward model, not from the policy. The real policy snapshot,
`π_old`, doesn't need a network of its own: it enters PPO as the **log-probabilities stored at
sampling time**, as the next cells show.

### Step 1 · Rollouts

Sample answers for a batch of prompts at temperature 1, so the samples come from the policy's own
distribution (PPO's math assumes this; Qwen3's default sampling settings would sharpen it). The
response mask marks the tokens the policy wrote, up to and including its first `<|im_end|>`;
everything after is padding. Answers that hit the 128-token limit are scored unfinished, a
simplification; real pipelines often penalize truncation.

In [ ]:
MAX_NEW = 128

@torch.no_grad()
def rollout(model, prompts, max_new=MAX_NEW):
    enc = tok([tok_prompt(q) for q in prompts], return_tensors="pt", padding=True, padding_side="left").to(DEVICE)
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=True, temperature=1.0, top_k=0, top_p=1.0,
                         pad_token_id=PAD)                 # sample from π itself, not Qwen3's default temperature 0.6
    P, resp = enc.input_ids.shape[1], out[:, enc.input_ids.shape[1]:]
    ended = (resp == IM_END) | (resp == PAD)
    last = torch.where(ended.any(1), ended.float().argmax(1), torch.full_like(ended[:, 0], resp.shape[1] - 1, dtype=torch.long))
    mask = (torch.arange(resp.shape[1], device=DEVICE)[None] <= last[:, None]).float()
    attn = torch.cat([enc.attention_mask, mask.long()], 1)
    pos = (attn.cumsum(1) - 1).clamp(min=0)               # left padding: positions must start at the first real token
    return {"ids": out, "attn": attn, "pos": pos, "P": P, "mask": mask, "last": last, "prompts": prompts,
            "truncated": ~ended.any(1)}

def token_logps(model, ro):
    """log π(a_t | s_t) for every response token: logits at position P-1+t predict token P+t."""
    logits = model(input_ids=ro["ids"], attention_mask=ro["attn"], position_ids=ro["pos"]).logits[:, ro["P"] - 1:-1].float()
    return torch.log_softmax(logits, -1).gather(-1, ro["ids"][:, ro["P"]:, None]).squeeze(-1)

def values_of(model, ro):
    """V(s_t) for every response position: the value head read at the token before a_t."""
    h = model.model(input_ids=ro["ids"], attention_mask=ro["attn"], position_ids=ro["pos"]).last_hidden_state
    return model.score(h).squeeze(-1).float()[:, ro["P"] - 1:-1] * ro["mask"]

@torch.no_grad()
def score_rollout(ro):
    return rm(input_ids=ro["ids"], attention_mask=ro["attn"], position_ids=ro["pos"]).logits.squeeze(-1).float()

torch.manual_seed(2)
ro = rollout(policy, rl_prompts[:8])
print(f"batch {tuple(ro['ids'].shape)} | prompt length P = {ro['P']} | response lengths {ro['mask'].sum(1).int().tolist()} | "
      f"truncated: {int(ro['truncated'].sum())}")

### Prediction

## Predict 12 · Where do rewards land?

The reward model gives one number per response. PPO also charges a KL penalty for moving away from `ref`. Build the per-token reward vector for one response: which positions are nonzero, and what is in each?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
BETA = 0.05

def ppo_rewards(R, logp, ref_logp, ro, beta=BETA):
    kl = (logp - ref_logp) * ro["mask"]                 # per-token log-ratio: a sample estimate of KL
    rewards = -beta * kl
    rewards[torch.arange(len(R)), ro["last"]] += R      # the reward-model score lands on the last token only
    return rewards, kl

with torch.no_grad():
    old_logp, ref_logp, R = token_logps(policy, ro), token_logps(ref, ro), score_rollout(ro)
rewards, kl = ppo_rewards(R, old_logp, ref_logp, ro)
k = int(ro["last"][0])
print(f"response 0: {k + 1} tokens | R = {R[0]:.3f}")
print("last 4 per-token rewards:", [round(v, 4) for v in rewards[0, k - 3:k + 1].tolist()])
print(f"max |reward| before the last token: {rewards[0, :k].abs().max():.2e} (all KL; policy == ref so far)")
assert torch.allclose(rewards[0, :k], torch.zeros(k, device=DEVICE), atol=1e-2)
torch.testing.assert_close(rewards[0, k], R[0], rtol=1e-2, atol=1e-2)

### Reference explanation

### Reference answer and mechanism

Every response token gets `−β · (log π − log π_ref)`, the per-token KL charge. The **last** token
also gets the reward-model score `R`. Nothing else carries reward: the reward model judged only the
finished answer (D20).

Right now `policy` and `ref` are identical, so every KL term is about 0 (bf16 noise) and the vector
is zero except at the end. As PPO trains, the policy drifts and the KL charges grow: a running cost
for every token where the policy has moved away from the reference.

### Prediction

## Predict 13 · GAE on the garlic example

Recall D24: values `[0.5, 0.1, 0.0, −1.8, −1.9]` for five positions and final reward −2.0 (KL ignored). Compute the advantages with λ = 0 and λ = 1. Which token gets the blame in each case?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
def gae(rewards, values, mask, gamma=1.0, lam=0.95):
    """Advantages A_t = Σ_k (γλ)^k δ_{t+k}, with δ_t = r_t + γ V(s_{t+1}) − V(s_t). Returns (A, return targets)."""
    T = rewards.shape[1]
    adv, running = torch.zeros_like(rewards), torch.zeros_like(rewards[:, 0])
    for t in reversed(range(T)):
        next_v = values[:, t + 1] * mask[:, t + 1] if t + 1 < T else torch.zeros_like(running)
        delta = rewards[:, t] + gamma * next_v - values[:, t]
        running = (delta + gamma * lam * running) * mask[:, t]
        adv[:, t] = running
    return adv, adv + values

V = torch.tensor([[0.5, 0.1, 0.0, -1.8, -1.9]]); r = torch.tensor([[0, 0, 0, 0, -2.0]]); m = torch.ones(1, 5)
tokens = ["No", ", raw garlic is", "toxic", ".", "<end>"]
for lam in (0.0, 1.0, 0.95):
    A, _ = gae(r, V, m, lam=lam)
    print(f"λ = {lam:4}: " + "  ".join(f"{t!r}: {a:+.2f}" for t, a in zip(tokens, A[0].tolist())))
torch.testing.assert_close(gae(r, V, m, lam=0.0)[0], torch.tensor([[-0.4, -0.1, -1.8, -0.1, -0.1]]))
torch.testing.assert_close(gae(r, V, m, lam=1.0)[0], torch.tensor([[-2.5, -2.1, -2.0, -0.2, -0.1]]))

### Reference explanation

### Reference answer and mechanism

- **λ = 0:** `A_t = δ_t`, the change in the value prediction caused by token t. "toxic" gets −1.8,
  most of the blame, because that is where the prediction collapsed.
- **λ = 1:** `A_t = R − V(s_t)`, the final outcome minus what was expected *before* token t. Every
  early token gets a large negative advantage (−2.5 for "No"); the blame isn't localized.
- **λ = 0.95** (the usual setting) sits in between.

Small λ trusts the value model more: low variance, but biased if V is wrong. Large λ trusts the
actual outcome more: unbiased, but noisy. The same function now runs on real rollouts.

In [ ]:
with torch.no_grad():
    V_real = values_of(value, ro)
A_real, _ = gae(rewards, V_real, ro["mask"])
k = int(ro["last"][0]) + 1
pieces = [tok.decode([t]) for t in ro["ids"][0, ro["P"]:ro["P"] + k]]
fig, axes = plt.subplots(2, 1, figsize=(12, 4.6), sharex=True)
axes[0].plot(V_real[0, :k].cpu(), color=BLUE, marker=".", label="V(s_t): value model")
axes[0].axhline(R[0].item(), color=ORANGE, linestyle="--", label=f"final reward-model score R = {R[0]:.2f}")
axes[0].legend(loc="best"); axes[0].set_ylabel("value")
axes[1].bar(range(k), A_real[0, :k].cpu(), color=[GREEN if a > 0 else RED for a in A_real[0, :k].tolist()])
axes[1].set_ylabel("advantage A_t (λ = 0.95)")
step = max(1, k // 30)
axes[1].set_xticks(range(0, k, step)); axes[1].set_xticklabels([repr(p)[1:-1][:8] for p in pieces[::step]], rotation=70, fontsize=7)
axes[0].set_title("One real rollout: value prediction at each token, and the advantage it produces")
plt.tight_layout(); plt.show()
print("PROMPT:", ro["prompts"][0])

### Reference explanation

### Reference answer and mechanism

The top panel is the value model's prediction at each prefix. Because `value` starts as a copy of
the reward model, `V(s_t)` here is roughly "the reward model's score of the answer so far". The
orange line is the score of the finished answer. The bottom panel shows the resulting advantages:
tokens where the prediction rose get positive credit, tokens where it fell get blame.

At this point the value model has never been trained as a value model; it is a reward model read at
every position. In the reference run its predictions swing by several points from one token to the
next: a reward model only ever learned to score **finished** answers, so its readings on half
sentences are noisy, and so are the advantages built from them. That noise is one reason PPO
trains the value model, and one reason GRPO's simpler group baseline works at all. During PPO it is regressed toward the observed returns, so its predictions become
"expected return of *this policy* from here" (D25).

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Value prediction and GAE advantage at every token of one real rollout](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-06.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Prediction

## Predict 14 · What does clipping do to the gradient?

The PPO loss per token is `−min(ρ·A, clip(ρ, 1−ε, 1+ε)·A)` with `ρ = π_θ/π_old`. With ε = 0.2, what is the gradient with respect to ρ when (a) ρ = 1.5, A = +1; (b) ρ = 1.5, A = −1; (c) ρ = 0.9, A = +1?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
EPS = 0.2

def ppo_policy_loss(logp, old_logp, adv, mask, eps=EPS):
    ratio = torch.exp(logp - old_logp)
    surrogate = torch.min(ratio * adv, ratio.clamp(1 - eps, 1 + eps) * adv)
    loss = -(surrogate * mask).sum() / mask.sum()
    clip_frac = (((ratio - 1).abs() > eps).float() * mask).sum() / mask.sum()
    return loss, clip_frac

for rho, a in [(1.5, 1.0), (1.5, -1.0), (0.9, 1.0)]:
    rho_t = torch.tensor(rho, requires_grad=True)
    loss = -torch.min(rho_t * a, rho_t.clamp(1 - EPS, 1 + EPS) * a)
    loss.backward()
    print(f"ρ = {rho}, A = {a:+.0f}: loss {loss.item():+.2f} | d loss / dρ = {rho_t.grad.item():+.1f}")

rho = torch.linspace(0.4, 1.8, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, a in zip(axes, (1.0, -1.0)):
    ax.plot(rho, rho * a, color=GRAY, linestyle="--", label="unclipped ρ·A")
    ax.plot(rho, torch.min(rho * a, rho.clamp(1 - EPS, 1 + EPS) * a), color=BLUE, label="PPO objective")
    ax.axvspan(1 - EPS, 1 + EPS, color=GREEN, alpha=0.08)
    ax.set_title(f"A = {a:+.0f}"); ax.set_xlabel("ratio ρ = π_θ / π_old"); ax.legend()
axes[0].set_ylabel("objective (maximized)")
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

- (a) ρ = 1.5, A = +1: the token is already 50% more likely than at sampling time. The objective is
  flat beyond 1 + ε, so the **gradient is 0**: no further push this round.
- (b) ρ = 1.5, A = −1: a bad token became *more* likely. The `min` picks the unclipped term, so the
  gradient is nonzero and pulls it back. Clipping never protects a move in the wrong direction.
- (c) ρ = 0.9, A = +1: inside the trust region; the ordinary gradient applies.

The ratio uses `π_old`: the log-probs stored when the batch was sampled (`old_logp`). Several
gradient steps reuse one batch, and clipping keeps them from moving any token's probability too far
from the policy that generated the data.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![PPO clipped objective against the probability ratio for positive and negative advantage](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-07.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

### Step 2 · The full PPO iteration

One iteration puts the pieces together. Each new piece is one line:

1. **rollout** with the current policy, and store `π_old` log-probs, `ref` log-probs and values
2. **score** with the frozen reward model and build per-token rewards (score + KL charges)
3. **GAE** → advantages and return targets
4. **update** the policy with the clipped loss, and the value model with `(V − return)²`, for a
   couple of epochs on the same batch

In [ ]:
N_PPO_ITERS, PPO_BS, PPO_EPOCHS = 40, 16, 2
policy_opt = torch.optim.AdamW(policy.parameters(), lr=3e-6)
value_opt = torch.optim.AdamW(value.parameters(), lr=5e-6)

def ppo_iteration(prompts):
    ro = rollout(policy, prompts)
    with torch.no_grad():
        old_logp, ref_logp, R, V = token_logps(policy, ro), token_logps(ref, ro), score_rollout(ro), values_of(value, ro)
    rewards, kl = ppo_rewards(R, old_logp, ref_logp, ro)
    adv, returns = gae(rewards, V, ro["mask"])
    m = ro["mask"].bool()
    adv = (adv - adv[m].mean()) / (adv[m].std() + 1e-8) * ro["mask"]          # whiten advantages per batch
    for _ in range(PPO_EPOCHS):
        p_loss, clip_frac = ppo_policy_loss(token_logps(policy, ro), old_logp, adv, ro["mask"])
        p_loss.backward(); torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); policy_opt.step(); policy_opt.zero_grad()
        v_loss = (((values_of(value, ro) - returns) ** 2) * ro["mask"]).sum() / ro["mask"].sum()
        v_loss.backward(); torch.nn.utils.clip_grad_norm_(value.parameters(), 1.0); value_opt.step(); value_opt.zero_grad()
    return {"reward": R.mean().item(), "kl": kl.sum(1).mean().item(), "value_loss": v_loss.item(),
            "clip_frac": clip_frac.item(), "length": ro["mask"].sum(1).mean().item()}

@torch.no_grad()
def eval_reward(model, seed=123):
    """Mean reward-model score on the fixed evaluation prompts, and its standard error."""
    torch.manual_seed(seed)
    R = torch.cat([score_rollout(rollout(model, eval_prompts[i:i + 16])) for i in range(0, len(eval_prompts), 16)])
    return R.mean().item(), (R.std() / len(R) ** 0.5).item()

fmt = lambda ms: f"{ms[0]:+.3f} ± {ms[1]:.3f}"

ppo_eval_before = eval_reward(policy)
torch.cuda.reset_peak_memory_stats(); random.seed(1); t0 = time.time(); ppo_log = []
for it in range(N_PPO_ITERS):
    ppo_log.append(ppo_iteration(random.sample(rl_prompts, PPO_BS)))
    if it % 10 == 9:
        recent = {k: np.mean([row[k] for row in ppo_log[-10:]]) for k in ppo_log[0]}
        print(f"iter {it + 1:3d} | " + " | ".join(f"{k} {v:.3f}" for k, v in recent.items()) + f" | {time.time() - t0:.0f}s")
ppo_peak = gib(torch.cuda.max_memory_allocated())
ppo_eval_after = eval_reward(policy)
print(f"eval reward ({len(eval_prompts)} fixed prompts, mean ± standard error): {fmt(ppo_eval_before)} → {fmt(ppo_eval_after)} | "
      f"peak GPU memory {ppo_peak:.1f} GiB")

### Prediction

## Predict 15 · What did 40 iterations do?

Before reading the plot: 40 iterations × 16 answers = 640 sampled answers. Should the mean reward-model score go up measurably? What should happen to the KL from `ref`, and to the value loss? Would a rising reward prove that the answers got better?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, key, color in zip(axes, ["reward", "kl", "value_loss", "length"], [BLUE, ORANGE, GREEN, GRAY]):
    series = np.array([row[key] for row in ppo_log])
    ax.plot(series, color=color, alpha=0.35)
    ax.plot(np.arange(4, len(series)), np.convolve(series, np.ones(5) / 5, mode="valid"), color=color)
    ax.set_title({"reward": "reward-model score", "kl": "KL from ref (per response)",
                  "value_loss": "value loss", "length": "response length (tokens)"}[key])
    ax.set_xlabel("PPO iteration")
plt.tight_layout(); plt.show()
first, last = np.mean([r["kl"] for r in ppo_log[:5]]), np.mean([r["kl"] for r in ppo_log[-5:]])
print(f"KL first 5 iterations {first:.3f} → last 5 {last:.3f}")
assert all(np.isfinite([r[k] for r in ppo_log for k in r]))

### Reference explanation

### Reference answer and mechanism

Read your own curves; with 40 small iterations they are noisy. Three things to check:

- **KL from `ref`** starts at 0 (the models are identical) and grows as the policy moves away. One
  batch's value can dip below 0, because it is a sample estimate from 16 responses, but its trend is
  the policy's distance from where it started.
- **Value loss** falls as the value model, started as a reward model, learns to predict *this
  policy's* returns (D25).
- **Reward-model score** is the noisiest curve, because each batch has different prompts. Use the
  fixed evaluation printed above, and compare the change with its **standard error**. In the
  reference run the change was within about one standard error: no measurable effect. 640 sampled
  answers and 40 small steps barely move a 0.6B policy; real RLHF runs use hundreds of thousands
  of samples. What did change measurably is the KL and the value loss: the machinery is working,
  the training is just very short.

So the honest reading is: **mechanics verified, no capability claim**. If you want to see the score
move, raise `N_PPO_ITERS` or the learning rate, and rerun the evaluation with its standard error.

Even a clearly higher reward-model score would **not** be evidence of better answers: it measures
agreement with a ~62%-accurate proxy that, as Part C showed, also responds to length and filler. Real RLHF watches KL,
length and held-out human or judge evaluations alongside the reward, for exactly this reason.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![PPO training curves: reward-model score, KL from reference, value loss and length](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-08.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

In [ ]:
del policy_opt, value_opt, value, policy
free()
print(f"after PPO: {gib(torch.cuda.memory_allocated()):.1f} GiB still allocated (reward model and ref)")

---

# Part F · GRPO: the group replaces the value model

GRPO samples **G answers per prompt**, scores them all, and uses each answer's score relative to its
own group as the advantage: `A_i = (R_i − mean(R)) / std(R)`. Every token of answer i shares `A_i`.
There is no value model (D21, D23).

### Prediction

## Predict 16 · Group advantages

Sample 8 answers to one prompt and score them. What do the 8 advantages sum to? And what happens if all 8 answers get the same score, as happens often in RLVR when a maths problem is too hard or too easy?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
G = 8

def group_advantages(R, G=G):
    R = R.view(-1, G)
    return ((R - R.mean(1, keepdim=True)) / (R.std(1, keepdim=True) + 1e-6)).flatten()

policy = load_chat_model()
torch.manual_seed(3)
ro = rollout(policy, [eval_prompts[1]] * G)
R = score_rollout(ro)
A = group_advantages(R)
print("PROMPT:", eval_prompts[1])
for i in R.argsort(descending=True).tolist():
    text = tok.decode(ro["ids"][i, ro["P"]:ro["P"] + int(ro["last"][i])], skip_special_tokens=True)
    print(f"R {R[i]:+.2f}  A {A[i]:+.2f}  {textwrap.shorten(text, 90)}")
print(f"sum of advantages: {A.sum():.2e}")
print("all scores equal →", group_advantages(torch.full((G,), 3.7, device=DEVICE)).tolist())
assert abs(A.sum().item()) < 1e-3

### Reference explanation

### Reference answer and mechanism

The advantages **sum to zero**: half the group is pushed up, half down, relative to the group's own
average. That average plays the role of `V(s_0)`, the baseline, estimated by sampling instead of by a
trained network. It is cheaper (no value model, no value loss) but cruder: every token in an answer
gets the same credit, with no per-token localization.

If all 8 scores are equal, every advantage is **0** and the prompt contributes no gradient. With a
verifier (RLVR), that happens whenever the model solves a problem in all 8 attempts or in none,
which is why GRPO pipelines filter for prompts of the right difficulty.

The GRPO loss is the same clipped surrogate as PPO, with the sequence-level advantage broadcast to
every token. The KL to `ref` moves out of the reward and into the loss, with the `k3` estimator
`exp(ref − logp) − (ref − logp) − 1`, which is always ≥ 0. Here each batch gets a single update, so `ρ = 1` at that step and
clipping is inactive; it matters when a batch is reused for several steps, as PPO did.

In [ ]:
N_GRPO_ITERS, GRPO_PROMPTS, GRPO_BETA = 40, 2, 0.04     # 2 prompts × G = 16 answers per iteration, as in PPO
grpo_opt = torch.optim.AdamW(policy.parameters(), lr=3e-6)

def grpo_iteration(prompts):
    ro = rollout(policy, [q for q in prompts for _ in range(G)])
    with torch.no_grad():
        old_logp, ref_logp, R = token_logps(policy, ro), token_logps(ref, ro), score_rollout(ro)
    adv = group_advantages(R)[:, None] * ro["mask"]                      # one advantage per answer, on every token
    logp = token_logps(policy, ro)
    clipped, _ = ppo_policy_loss(logp, old_logp, adv, ro["mask"])
    d = ref_logp - logp
    kl = ((torch.exp(d) - d - 1) * ro["mask"]).sum() / ro["mask"].sum()
    loss = clipped + GRPO_BETA * kl
    loss.backward(); torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); grpo_opt.step(); grpo_opt.zero_grad()
    return {"reward": R.mean().item(), "kl": kl.item(), "zero_adv_groups": int((R.view(-1, G).std(1) < 1e-3).sum())}

grpo_eval_before = eval_reward(policy)
torch.cuda.reset_peak_memory_stats(); random.seed(2); t0 = time.time(); grpo_log = []
for it in range(N_GRPO_ITERS):
    grpo_log.append(grpo_iteration(random.sample(rl_prompts, GRPO_PROMPTS)))
    if it % 10 == 9:
        print(f"iter {it + 1:3d} | reward {np.mean([r['reward'] for r in grpo_log[-10:]]):.3f} | "
              f"kl {np.mean([r['kl'] for r in grpo_log[-10:]]):.4f} | {time.time() - t0:.0f}s")
grpo_peak = gib(torch.cuda.max_memory_allocated())
grpo_eval_after = eval_reward(policy)
print(f"eval reward: {fmt(grpo_eval_before)} → {fmt(grpo_eval_after)} | peak GPU memory: GRPO {grpo_peak:.1f} GiB vs PPO {ppo_peak:.1f} GiB")

### Prediction

## Predict 17 · PPO vs. GRPO in memory

Both sample 16 answers per iteration (PPO: 16 prompts × 1; GRPO: 2 prompts × 8) of the same maximum length. Which used more peak GPU memory, and what accounts for the difference?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot([r["reward"] for r in grpo_log], color=BLUE, marker=".")
axes[0].set_title("GRPO: mean reward-model score per batch"); axes[0].set_xlabel("iteration")
axes[1].bar(["PPO\n(policy+ref+RM+value)", "GRPO\n(policy+ref+RM)"], [ppo_peak, grpo_peak], color=[ORANGE, BLUE])
axes[1].set_ylabel("peak GPU memory (GiB)"); axes[1].set_title("peak memory during training")
plt.tight_layout(); plt.show()
print(f"PPO {ppo_peak:.1f} GiB | GRPO {grpo_peak:.1f} GiB")
assert grpo_peak < ppo_peak

### Reference explanation

### Reference answer and mechanism

With the same 16 answers per iteration, GRPO's peak is lower. PPO trains **two** networks, the
policy and the value model, each with weights, gradients and two AdamW buffers (all bf16 here:
about 8 bytes per parameter, roughly 4–5 GiB for a 0.6B value model), plus the value model's
activations.
GRPO drops all of that. At 70B, the value model costs as much as the policy, which is a large part
of GRPO's appeal.

What GRPO pays instead is **samples per prompt**: 8 answers to one prompt to estimate the baseline
a value model would have predicted, so with the same budget it sees fewer distinct prompts. At this
small scale, activations matter too: a batch of 16 sequences of 150 tokens produces float32 logits
over a 151,936-word vocabulary, about 1.5 GB per copy. Doubling the batch would erase GRPO's saving
here, which is why the comparison keeps the number of answers equal.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![GRPO reward per batch, and peak GPU memory of PPO and GRPO](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-09.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

In [ ]:
del grpo_opt, policy
free()

---

# Part G · DPO: the same pairs, no reward model, no sampling

DPO trains the chat model **directly on preference pairs**, the same kind of data the reward model
used (D19). Its "score" for an answer is how much more likely the policy makes it than the reference
does: `r̂(x, y) = β · [log π_θ(y|x) − log π_ref(y|x)]`, summed over the answer's tokens. The loss is
Bradley–Terry on that implicit score: `−log σ(r̂(x, y_chosen) − r̂(x, y_rejected))` (D21).

In [ ]:
DPO_BETA, DPO_MAXLEN = 0.1, 320

def answer_logps(model, pairs):
    """Sum of log π(answer tokens + <|im_end|>) for chosen and rejected answers, in one forward pass. Returns [B, 2]."""
    seqs, starts = [], []
    for p in pairs:
        prompt_ids = tok(tok.apply_chat_template(p["prompt"], add_generation_prompt=True, tokenize=False,
                                                 enable_thinking=False), add_special_tokens=False).input_ids
        for ans in (p["chosen"], p["rejected"]):
            ans_ids = tok(ans + "<|im_end|>", add_special_tokens=False).input_ids
            seq = (prompt_ids + ans_ids)[-DPO_MAXLEN:]
            seqs.append(seq); starts.append(max(1, len(seq) - len(ans_ids)))
    L = max(map(len, seqs))
    ids = torch.full((len(seqs), L), PAD); attn = torch.zeros_like(ids); amask = torch.zeros(len(seqs), L)
    for i, (s, st) in enumerate(zip(seqs, starts)):
        ids[i, :len(s)] = torch.tensor(s); attn[i, :len(s)] = 1; amask[i, st:len(s)] = 1
    ids, attn, amask = ids.to(DEVICE), attn.to(DEVICE), amask.to(DEVICE)
    logits = model(input_ids=ids, attention_mask=attn).logits[:, :-1].float()
    lp = torch.log_softmax(logits, -1).gather(-1, ids[:, 1:, None]).squeeze(-1)
    return (lp * amask[:, 1:]).sum(1).view(-1, 2)

def dpo_loss(policy_lp, ref_lp, beta=DPO_BETA):
    implicit = beta * (policy_lp - ref_lp)                    # [B, 2]: implicit reward of chosen, rejected
    return -F.logsigmoid(implicit[:, 0] - implicit[:, 1]).mean(), implicit

dpo_pairs = rm_train_pairs[:1600]          # literally the same pairs the reward model trained on
print(f"DPO trains on {len(dpo_pairs)} pairs; same objects as the reward model's data: {dpo_pairs[0] is rm_train_pairs[0]}")

### Prediction

## Predict 18 · DPO at step 0

The policy starts as an exact copy of `ref`. Before any update, what is the implicit reward of every answer, and what is the DPO loss?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
policy = load_chat_model()
with torch.no_grad():
    loss0, implicit0 = dpo_loss(answer_logps(policy, dpo_pairs[:8]), answer_logps(ref, dpo_pairs[:8]))
print(f"implicit rewards (max |value|): {implicit0.abs().max():.2e} | loss {loss0:.4f} | log 2 = {math.log(2):.4f}")
assert abs(loss0.item() - math.log(2)) < 0.02

### Reference explanation

### Reference answer and mechanism

Every implicit reward is **0** (up to bf16 noise), because `π_θ = π_ref` makes every log-ratio 0. The
loss is `−log σ(0) = log 2 ≈ 0.693`, the same "chance" value as an untrained reward model in
Part C. DPO is Bradley–Terry training where the score is *defined* by how far the policy has moved
from the reference, so the only way to lower the loss is to raise chosen answers' probability
relative to rejected ones, measured against `ref`.

No sampling happens: both answers come from the dataset (offline), and the reference's log-probs
replace a reward model.

In [ ]:
DPO_BS = 8
dpo_opt = torch.optim.AdamW(policy.parameters(), lr=3e-6)

@torch.no_grad()
def dpo_eval(pairs):
    policy.eval(); out = []
    for i in range(0, len(pairs), 8):
        _, imp = dpo_loss(answer_logps(policy, pairs[i:i + 8]), answer_logps(ref, pairs[i:i + 8]))
        out.append(imp)
    policy.train(); imp = torch.cat(out)
    return (imp[:, 0] > imp[:, 1]).float().mean().item(), imp

dpo_eval_pairs = test_pairs[:300]
torch.cuda.reset_peak_memory_stats(); policy.train(); dpo_log, t0 = [], time.time()
for i in range(0, len(dpo_pairs), DPO_BS):
    batch = dpo_pairs[i:i + DPO_BS]
    with torch.no_grad():
        ref_lp = answer_logps(ref, batch)
    loss, implicit = dpo_loss(answer_logps(policy, batch), ref_lp)
    loss.backward(); torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); dpo_opt.step(); dpo_opt.zero_grad()
    dpo_log.append({"loss": loss.item(), "margin": (implicit[:, 0] - implicit[:, 1]).mean().item(),
                    "chosen": implicit[:, 0].mean().item(), "rejected": implicit[:, 1].mean().item()})
    if len(dpo_log) % 50 == 0:
        print(f"step {len(dpo_log):3d} | loss {np.mean([r['loss'] for r in dpo_log[-50:]]):.3f} | "
              f"margin {np.mean([r['margin'] for r in dpo_log[-50:]]):+.3f} | {time.time() - t0:.0f}s")
dpo_peak = gib(torch.cuda.max_memory_allocated())
dpo_acc, dpo_imp = dpo_eval(dpo_eval_pairs)
print(f"held-out implicit-reward accuracy: {dpo_acc:.1%} | peak GPU memory {dpo_peak:.1f} GiB")
tail = dpo_log[-50:]
print(f"last 50 steps, mean implicit reward: chosen {np.mean([r['chosen'] for r in tail]):+.3f} | "
      f"rejected {np.mean([r['rejected'] for r in tail]):+.3f}")

### Prediction

## Predict 19 · Two learners, one dataset

The reward model (Part C) and DPO learned from the same kind of pairs. On held-out pairs, how often will they agree about which answer is better? And during DPO training, do you expect the chosen answers' implicit reward to go up, or the rejected answers' to go down?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
with torch.no_grad():
    _, rm_c, rm_r = pair_accuracy(rm, dpo_eval_pairs)
rm_pref = (rm_c > rm_r).cpu(); dpo_pref = (dpo_imp[:, 0] > dpo_imp[:, 1]).cpu()
agree = (rm_pref == dpo_pref).float().mean().item()
print(f"reward model accuracy {rm_pref.float().mean():.1%} | DPO implicit accuracy {dpo_pref.float().mean():.1%} | "
      f"they agree on {agree:.1%} of held-out pairs")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for key, color, label in [("chosen", GREEN, "chosen"), ("rejected", RED, "rejected")]:
    s = np.array([r[key] for r in dpo_log])
    axes[0].plot(np.convolve(s, np.ones(10) / 10, mode="valid"), color=color, label=f"implicit reward: {label}")
axes[0].axhline(0, color=GRAY, linewidth=0.8); axes[0].legend(); axes[0].set_xlabel("DPO step (10-step mean)")
axes[0].set_title("where does the margin come from?")
axes[1].scatter((rm_c - rm_r).cpu(), (dpo_imp[:, 0] - dpo_imp[:, 1]).cpu(), s=8, color=BLUE, alpha=0.5)
axes[1].axhline(0, color=GRAY, linewidth=0.8); axes[1].axvline(0, color=GRAY, linewidth=0.8)
axes[1].set_xlabel("reward-model gap"); axes[1].set_ylabel("DPO implicit gap"); axes[1].set_title(f"agreement {agree:.0%}")
plt.tight_layout(); plt.show()
assert dpo_acc > 0.5

### Reference explanation

### Reference answer and mechanism

**Agreement.** Both learners beat chance on held-out pairs, but they agree on a pair less often than
you might expect from "same data". They are different models with different inductive biases: one
is a trained scalar head reading the whole answer, the other is a shift in the policy's token
probabilities. Their errors are only partly shared. "Same data" doesn't mean "same learned taste".

**Where the margin comes from.** DPO only controls the *difference* between the two implicit
rewards. In the reference run both rose (chosen about +0.28, rejected about +0.21, last 50 steps),
with chosen rising faster: the policy made *both* dataset answers more likely and favoured the
chosen one. Other runs and datasets often show the opposite: both fall, rejected faster. That is a
known DPO side effect: a wider margin with the chosen answers themselves becoming *less* likely.
The loss can't tell these cases apart; only the two curves can.

**Offline vs. online (D14).** DPO never saw an answer it wrote itself. Every gradient came from the
fixed dataset, which was written by other models. PPO and GRPO learned only from their own samples.
That's the main trade-off between the recipes, not the data format.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![DPO implicit rewards during training, and agreement with the reward model](../figures/chapter-10/day-15-04_preference_alignment_end_to_end-10.png)

This fixed image comes from the Spark GPU reference run; changing inputs does not update the preview automatically.

## Reading the evidence

| | Reward model (Part C) | PPO (Part E) | GRPO (Part F) | DPO (Part G) |
|---|---|---|---|---|
| Training data | labeled pairs | prompts only | prompts only | labeled pairs |
| Who labels during training | the dataset | frozen reward model | frozen reward model (or a verifier) | the dataset |
| Generates during training | no | yes, 1 per prompt | yes, G per prompt | no |
| Baseline for "better than expected" | — | value model, per token | group mean, per answer | the reference model |
| Networks in memory | 1 | 4 (2 trained) | 3 (1 trained) | 2 (1 trained) |

**What this notebook establishes, for this run:** clean preference triples come from shared-prefix
transcripts; real labels include ties and small margins; chosen answers are longer more often. A
reward model is a pretrained transformer with a 1-output head, read at the last token. Pairwise
training lifts held-out accuracy from chance to the 60s, and its score also responds to length and
filler. In PPO, the reward-model score lands on the last token, KL charges on every token, GAE turns
value predictions into per-token advantages, and clipping zeroes the gradient once a token has moved
far enough in the right direction. GRPO replaces the value model with a group mean and uses less
memory. DPO starts at loss `log 2` and learns from the same pairs with no reward model.

**What it doesn't establish:** that any trained policy is better aligned. Runs are tens of
iterations on a 0.6B model, the reward model is about 62% accurate, and its score is a proxy; the
PPO and GRPO evaluation changes were within about one standard error.
Rewards, KL values and agreements vary between runs. The comparisons of memory and data are
mechanical facts about these implementations, not benchmarks of the methods.

**Where to go next:** Chapter 10 (reward models), Chapter 11 (DPO), Chapters 12–13 (policy gradients,
PPO, GRPO), Chapter 14 (reward hacking, plan P2). The
[learning note](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) holds the discussions behind each part.